# Recherche d'information dans la presse française de 1914

## Récupération des données


In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914

In [ ]:
!ls french-press-1914/le_figaro/1914/

## Corpus

Ce corpus est composé d'articles de la presse française de l'année 1914.

Un dossier par journal, un dossier contenant un fichier JSON par date de parution.

Les fichiers JSON contiennent plusieurs champs, dont trois qui nous intéresseront pour la suite de ce travail :

- `contentAsText`, une liste de chaînes de caractères (les paragraphes sont délimités dans ces chaînes par des sauts de ligne — `\n`)
- `date`, la date de parution du journal, représentée par une liste contenant une chaîne de caractères (si la variable `content` contient le fichier JSON chargé, on y accède par `content["date"][0]`)
- `title`, le nom du journal, représenté par une liste contenant une chaîne de caractères (si la variable `content` contient le fichier JSON chargé, on y accède par `content["title"][0]`)

Implémentez une fonction qui prend en entrée le chemin du corpus et qui retourne un tableau `numpy.ndarray` à 3 colonnes :

- la première colonne contient les paragraphes des éditions du corpus (on a donc une ligne dans le tableau par paragraphe du corpus), avec respect des conditions suivantes :
  - seulement les paragraphes de plus de 10 mots
  - casse minuscule
  - sans signes de ponctuation
- la deuxième colonne contient le journal dont le paragraphe est issu
- la troisième colonne contient la date de l'édition dont le paragraphe est issu

In [ ]:
# Votre code ici

### Solution

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def preprocess_text(text: str) -> str:
  out = re.sub(r"\W+", " ", text)
  return out.lower()


def get_data(directory: pathlib.Path) -> numpy.ndarray:
  data = []
  for edition in tqdm.notebook.tqdm(list(directory.rglob("*.json"))):
    with open(edition, encoding="utf8") as fh:
      content = json.load(fh)
    newspaper = content["title"][0]
    date = datetime.date.fromisoformat(content["date"][0]).strftime("%d/%m/%Y")
    for item in content["contentAsText"]:
      for paragraph in item.split("\n"):
        processed_paragraph = preprocess_text(paragraph)
        if len(processed_paragraph.split()) > 10:
          data.append((processed_paragraph, newspaper, date))
  return numpy.array(data)


data = get_data(pathlib.Path("french-press-1914"))

## Représentation TF-IDF

Utilisez un [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) afin de créer la matrice `X` des paragraphes du corpus transformés en vecteurs de poids TF-IDF.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data[:, 0])

## Similarité

Implémentez une fonction `best_score(query: str) -> None` qui :

- prend en entrée une requête sous forme de chaîne de caractères
- affiche les journaux et dates des 5 meilleurs résultats au sens de la similarité cosinus

In [ ]:
# Votre code ici

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  query_vector = vectorizer.transform([query.lower()])
  cosine = sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]
  # On utilise l'opposé du tableau cosine dans l'appel à argsort afin d'obtenir
  # les indices du tableau triés par valeurs décroissantes avant d'en récupérer
  # les 5 premiers
  top5 = (-cosine).argsort()[:5]
  for index, (text, newspaper, date), similarity in zip(top5, data[top5], cosine[top5]):
    print(
      f"{newspaper:10} du {date} ; "
      f"similarité {similarity:5.2f} ; "
      f"index {index:6} ; "
      f"{text}"
    )

## BM25

Essayez d'abord `best_score("guerre allemands")` : que remarquez-vous sur la longueur des paragraphes en tête ?

Le cours présente Okapi BM25, une amélioration du produit scalaire TF-IDF, qui compare la longueur de chaque document à la longueur moyenne :

$$
\text{score}(D, Q)
= \sum_{i = 1}^n
\text{IDF}(q_i)
\cdot
\frac{f(q_i, D) \cdot (k_1 + 1)}
     {f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\bar{|D|}}\right)}
$$

où :

- $q_1, \dots, q_n$ sont les termes de la requête $Q$
- $\text{IDF}(q_i) = \log\frac{N}{df_i}$, celui de TF-IDF : $df_i$ documents sur les $N$ de la base contiennent $q_i$
- $f(q_i, D)$ est la fréquence du terme $q_i$ dans le document $D$
- $|D|$ est le nombre de termes de $D$, et $\bar{|D|}$ la longueur moyenne des documents de la base
- $k_1 = 1{,}5$ et $b = 0{,}75$

Ici, un document est un paragraphe. Comptez les termes de chaque paragraphe avec un `CountVectorizer`, dans une variable `compteur` et une matrice `comptes`, et gardez la longueur de chaque paragraphe dans `longueurs`. Implémentez ensuite la fonction `scores_bm25`, qui renvoie le score de chaque paragraphe pour une requête.

**Documentation utile**

- [`sklearn.feature_extraction.text.CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [`scipy.sparse.csr_matrix.getnnz`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.getnnz.html)
- [`numpy.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

In [ ]:
# Votre code ici


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `compteur.fit_transform(data[:, 0])` : la matrice creuse `comptes`, avec une ligne par paragraphe et une colonne par terme
- `comptes.getnnz(axis=0)` : pour chaque terme, le nombre de paragraphes qui le contiennent ($df_i$)
- `numpy.asarray(comptes.sum(axis=1)).ravel()` : pour chaque paragraphe, son nombre de termes ($|D|$). `CountVectorizer` ignore les mots d'une lettre : 12 paragraphes n'ont donc aucun terme, comptez-les pour 1 (`numpy.maximum(..., 1)`), sinon $b = 1$ divise par zéro
- `compteur.transform([requete]).indices` : les colonnes des termes de la requête. `CountVectorizer` la met en minuscules, la découpe et ignore les mots absents du corpus
- `comptes[:, terme].toarray().ravel()` : la colonne d'un terme, soit $f(q_i, D)$ pour tous les paragraphes. Les opérations de numpy portent alors sur tous les paragraphes à la fois, sans boucle sur les documents
- pour vérifier : `N` vaut 48 333, `longueurs.mean()` environ 178 et `scores_bm25("jaurès assassiné").max()` environ 13,51

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
compteur = sklearn.feature_extraction.text.CountVectorizer()
comptes = ...
N = comptes.shape[0]
df = ...  # df_i pour chaque terme
longueurs = numpy.maximum(..., 1)  # |D| pour chaque paragraphe


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # La partie du dénominateur qui ne dépend que du paragraphe
  normalisation = k1 * (1 - b + b * ...)
  scores = numpy.zeros(N)
  for terme in compteur.transform([requete]).indices:
    f = ...  # f(q_i, D) pour tous les paragraphes
    idf = ...
    scores += ...
  return scores
```

### Solution

In [ ]:
best_score("guerre allemands")

compteur = sklearn.feature_extraction.text.CountVectorizer()
comptes = compteur.fit_transform(data[:, 0])
N = comptes.shape[0]
df = comptes.getnnz(axis=0)
# CountVectorizer ignore les mots d'une lettre : 12 paragraphes n'ont aucun terme, et on
# les compte pour 1 pour ne pas diviser par zéro quand b vaut 1
longueurs = numpy.maximum(numpy.asarray(comptes.sum(axis=1)).ravel(), 1)


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  normalisation = k1 * (1 - b + b * longueurs / longueurs.mean())
  scores = numpy.zeros(N)
  for terme in compteur.transform([requete]).indices:
    f = comptes[:, terme].toarray().ravel()
    idf = numpy.log(N / df[terme])
    scores += idf * f * (k1 + 1) / (f + normalisation)
  return scores


print(scores_bm25("jaurès assassiné").max())

## Comparaison avec le cosinus

Implémentez une fonction `compare(requete: str) -> None` qui affiche, pour la similarité cosinus puis pour BM25, les 5 meilleurs résultats : le score, la longueur, l'index et le début du texte. Le texte entier d'un résultat se lit dans `data[index, 0]`.

Regardez ensuite :

1. sur `guerre allemands` et `jaurès assassiné`, la longueur des paragraphes que chaque méthode met en tête, par rapport à la longueur médiane du corpus. Laquelle remonte l'article sur l'assassinat de Jaurès ?
2. la longueur des 5 premiers résultats de BM25 sur `guerre allemands`, avec $b$ valant 0, puis 0,75, puis 1 : que règle $b$ ?
3. sur `mobilisation générale`, les résultats des deux méthodes : combien annoncent vraiment une mobilisation générale ? Pourquoi ?

Concluez : qu'apporte la normalisation par la longueur, et que ne résout-elle pas ?

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)

In [ ]:
def compare(requete: str) -> None:
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `(-scores).argsort()[:5]` : les indices des 5 meilleurs scores, comme dans `best_score`
- `sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0]` : la similarité de chaque paragraphe avec la requête
- `data[index, 0][:80]` : le début d'un paragraphe
- `scores_bm25("guerre allemands", b=b)` : BM25 avec une autre valeur de $b$
- pour vérifier : `numpy.median(longueurs)` vaut 102

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def affiche_top5(scores: numpy.ndarray) -> None:
  for index in ...:  # les indices des 5 meilleurs scores
    debut = data[index, 0][:80]
    print(f"{scores[index]:5.2f} ; {longueurs[index]:4} termes ; {index:6} ; {debut}")


def compare(requete: str) -> None:
  print(f"Requête : {requete}")
  print("Cosinus")
  vecteur_requete = vectorizer.transform([requete])
  affiche_top5(...)  # la similarité de chaque paragraphe avec la requête
  print("BM25")
  affiche_top5(...)


print("Longueur médiane :", ...)
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  meilleurs = ...  # les indices des 5 meilleurs scores de BM25 avec ce b
  print(f"b = {b} : {longueurs[meilleurs]}")

compare("mobilisation générale")
```

### Solution

In [ ]:
def affiche_top5(scores: numpy.ndarray) -> None:
  for index in (-scores).argsort()[:5]:
    debut = data[index, 0][:80]
    print(f"{scores[index]:5.2f} ; {longueurs[index]:4} termes ; {index:6} ; {debut}")


def compare(requete: str) -> None:
  print(f"Requête : {requete}")
  print("Cosinus")
  vecteur_requete = vectorizer.transform([requete])
  affiche_top5(sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0])
  print("BM25")
  affiche_top5(scores_bm25(requete))


print("Longueur médiane :", numpy.median(longueurs))
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  meilleurs = (-scores_bm25("guerre allemands", b=b)).argsort()[:5]
  print(f"b = {b} : {longueurs[meilleurs]}")

compare("mobilisation générale")

Ce que montrent ces résultats :

- La similarité cosinus divise par la norme du vecteur du paragraphe : dans un titre, les termes de la requête pèsent lourd. Sur `guerre allemands`, ses 5 premiers résultats font 12 à 14 termes, pour une médiane de 102. Sur `jaurès assassiné`, elle met en tête un marchand de vins (10 termes) et l'article sur Jaurès (691 termes) n'arrive que 3e.
- BM25 met cet article en tête, et les quatre suivants parlent aussi de la mort de Jaurès. Sur `guerre allemands`, deux paragraphes de 84 et 85 termes entrent dans les 5 premiers, mais trois titres restent : les textes courts sont moins favorisés, ils ne sont pas écartés.
- $b$ règle cette normalisation : à 0, les 5 premiers font 656 à 1 042 termes, car un long texte contient plus souvent les termes ; à 1, BM25 retrouve des titres de 12 à 14 termes, comme le cosinus.
- La normalisation ne règle pas tout : sur `mobilisation générale`, un seul résultat sur 5 l'annonce vraiment (celle de la Hollande), dans les deux méthodes. Une grève générale en Afrique du Sud, avec une mobilisation de troupes, arrive 1re au cosinus et 2e avec BM25 : un sac de mots ne sait pas que les deux termes doivent se suivre. Le récit de l'affichage de l'ordre à Paris n'arrive que 6e avec BM25, 16e au cosinus.

Trois requêtes lues à la main ne mesurent rien : pour comparer deux moteurs, il faut des jugements de pertinence.